In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
import sqlite3
import pandas as pd

# Ruta correcta con espacios exactos
ruta_csv = '/content/drive/MyDrive/Proyecto_MuniMetro_Lima /02_Datos_Limpios /lima_metropolitana_final.csv'
df = pd.read_csv(ruta_csv)

# Crear base de datos en Drive
conn = sqlite3.connect('/content/drive/MyDrive/Proyecto_MuniMetro_Lima /02_Datos_Limpios /muni_lima.db')

# Cargar como tabla SQL
df.to_sql('ejecucion_presupuestal', conn, if_exists='replace', index=False)

# Verificar
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM ejecucion_presupuestal")
print("Filas cargadas:", cursor.fetchone()[0])

cursor.execute("PRAGMA table_info(ejecucion_presupuestal)")
print("\nColumnas:")
for col in cursor.fetchall():
    print(" -", col[1], "→", col[2])

Filas cargadas: 172

Columnas:
 - Municipalidad → TEXT
 - PIA → INTEGER
 - PIM → INTEGER
 - Certificacion → INTEGER
 - Compromiso_Anual → INTEGER
 - Atencion_Compromiso → INTEGER
 - Devengado → INTEGER
 - Girado → INTEGER
 - Avance → REAL
 - año → INTEGER
 - pct_ejecucion → REAL
 - distrito_limpio → TEXT
 - brecha_inversion → INTEGER


In [9]:
# ============================================
# CONSULTA 1: Ranking general por año
# ¿Cuál fue el top 5 de cada año?
# ============================================

query1 = """
SELECT
    año,
    distrito_limpio,
    pct_ejecucion,
    RANK() OVER (PARTITION BY año ORDER BY pct_ejecucion DESC) AS ranking
FROM ejecucion_presupuestal
ORDER BY año, ranking
LIMIT 20;
"""

df1 = pd.read_sql_query(query1, conn)
print("TOP 5 POR AÑO")
print("=" * 50)
print(df1.to_string(index=False))

TOP 5 POR AÑO
 año      distrito_limpio  pct_ejecucion  ranking
2023          JESUS MARIA           93.7        1
2023           LURIGANCHO           92.8        2
2023          LA VICTORIA           92.4        3
2023         PUEBLO LIBRE           92.1        4
2023                ANCON           92.1        4
2023           CHORRILLOS           92.0        6
2023                LURIN           92.0        6
2023           SAN MIGUEL           91.7        8
2023    MAGDALENA DEL MAR           90.7        9
2023           LOS OLIVOS           90.4       10
2023            LA MOLINA           89.1       11
2023                COMAS           88.8       12
2023          SANTA ANITA           88.6       13
2023    SANTIAGO DE SURCO           87.7       14
2023 SAN MARTIN DE PORRES           87.7       14
2023        INDEPENDENCIA           87.4       16
2023            SURQUILLO           87.3       17
2023                LINCE           84.6       18
2023                  ATE           

In [10]:
# ============================================
# CONSULTA 2: Promedio 2023-2025 por distrito
# (excluye 2026 por ser dato parcial)
# ============================================

query2 = """
SELECT
    distrito_limpio,
    ROUND(AVG(pct_ejecucion), 1) AS promedio_2023_2025,
    MIN(pct_ejecucion) AS minimo,
    MAX(pct_ejecucion) AS maximo
FROM ejecucion_presupuestal
WHERE año IN (2023, 2024, 2025)
GROUP BY distrito_limpio
ORDER BY promedio_2023_2025 DESC;
"""

df2 = pd.read_sql_query(query2, conn)
print("RANKING PROMEDIO 2023-2025")
print("=" * 55)
print(df2.to_string(index=False))

RANKING PROMEDIO 2023-2025
        distrito_limpio  promedio_2023_2025  minimo  maximo
             LURIGANCHO                97.2    92.8   100.0
             SAN MIGUEL                94.6    91.7    98.1
            JESUS MARIA                94.6    93.7    95.7
            LA VICTORIA                94.5    92.4    97.1
              LA MOLINA                93.5    89.1    97.7
           PUEBLO LIBRE                93.4    92.1    94.9
             LOS OLIVOS                93.0    90.4    95.6
      SANTIAGO DE SURCO                92.2    87.7    96.1
                  LURIN                91.3    90.0    92.0
      MAGDALENA DEL MAR                91.2    90.1    92.9
              SURQUILLO                90.7    87.3    92.5
                  COMAS                90.6    87.3    95.6
                  LINCE                89.9    84.6    95.0
                  ANCON                89.7    88.0    92.1
            SANTA ANITA                89.5    86.4    93.5
    SANTA MAR

In [12]:
# ============================================
# CONSULTA 3: Distritos que NUNCA bajaron del 80%
# en 2023, 2024 y 2025 (eficientes consistentes)
# ============================================

query3 = """
SELECT
    distrito_limpio,
    SUM(CASE WHEN pct_ejecucion >= 80 THEN 1 ELSE 0 END) AS años_sobre_meta,
    ROUND(AVG(pct_ejecucion), 1) AS promedio
FROM ejecucion_presupuestal
WHERE año IN (2023, 2024, 2025)
GROUP BY distrito_limpio
HAVING años_sobre_meta = 3
ORDER BY promedio DESC;
"""

df3 = pd.read_sql_query(query3, conn)
print("EFICIENTES CONSISTENTES (3 años ≥ 80%)")
print("=" * 45)
print(f"Total: {len(df3)} distritos")
print()
print(df3.to_string(index=False))

EFICIENTES CONSISTENTES (3 años ≥ 80%)
Total: 29 distritos

       distrito_limpio  años_sobre_meta  promedio
            LURIGANCHO                3      97.2
            SAN MIGUEL                3      94.6
           JESUS MARIA                3      94.6
           LA VICTORIA                3      94.5
             LA MOLINA                3      93.5
          PUEBLO LIBRE                3      93.4
            LOS OLIVOS                3      93.0
     SANTIAGO DE SURCO                3      92.2
                 LURIN                3      91.3
     MAGDALENA DEL MAR                3      91.2
             SURQUILLO                3      90.7
                 COMAS                3      90.6
                 LINCE                3      89.9
                 ANCON                3      89.7
           SANTA ANITA                3      89.5
   SANTA MARIA DEL MAR                3      89.3
         INDEPENDENCIA                3      89.3
             SAN BORJA                3 

In [13]:
# ============================================
# CONSULTA 4: Distritos con mayor mejora
# entre 2023 y 2025
# ============================================

query4 = """
SELECT
    a.distrito_limpio,
    a.pct_ejecucion AS ejecucion_2023,
    b.pct_ejecucion AS ejecucion_2025,
    ROUND(b.pct_ejecucion - a.pct_ejecucion, 1) AS mejora_puntos
FROM ejecucion_presupuestal a
JOIN ejecucion_presupuestal b
    ON a.distrito_limpio = b.distrito_limpio
WHERE a.año = 2023 AND b.año = 2025
ORDER BY mejora_puntos DESC;
"""

df4 = pd.read_sql_query(query4, conn)
print("RANKING DE MEJORA 2023 → 2025")
print("=" * 50)
print(df4.to_string(index=False))

RANKING DE MEJORA 2023 → 2025
        distrito_limpio  ejecucion_2023  ejecucion_2025  mejora_puntos
VILLA MARIA DEL TRIUNFO            53.9            92.7           38.8
             MIRAFLORES            62.4            93.6           31.2
               BARRANCO            72.7            97.3           24.6
 SAN JUAN DE LURIGANCHO            75.3            97.4           22.1
      VILLA EL SALVADOR            67.0            89.1           22.1
          PUNTA HERMOSA            55.3            77.2           21.9
            PUNTA NEGRA            73.8            95.2           21.4
          PUENTE PIEDRA            80.7            98.7           18.0
                  RIMAC            78.1            92.0           13.9
            CIENEGUILLA            79.1            92.8           13.7
              SAN BORJA            82.8            95.7           12.9
 SAN JUAN DE MIRAFLORES            82.8            95.7           12.9
               SAN LUIS            81.9        

In [14]:
# ============================================
# CONSULTA 5: Brecha de inversión
# PIM no ejecutado (dinero que se perdió)
# ============================================

query5 = """
SELECT
    distrito_limpio,
    año,
    PIM,
    Devengado,
    (PIM - Devengado) AS soles_no_ejecutados,
    pct_ejecucion
FROM ejecucion_presupuestal
WHERE año IN (2023, 2024, 2025)
ORDER BY soles_no_ejecutados DESC
LIMIT 15;
"""

df5 = pd.read_sql_query(query5, conn)
df5['PIM'] = df5['PIM'].apply(lambda x: f"S/ {x:,.0f}")
df5['Devengado'] = df5['Devengado'].apply(lambda x: f"S/ {x:,.0f}")
df5['soles_no_ejecutados'] = df5['soles_no_ejecutados'].apply(lambda x: f"S/ {x:,.0f}")

print("TOP 15 — MAYOR BRECHA DE INVERSIÓN (soles no ejecutados)")
print("=" * 70)
print(df5.to_string(index=False))

TOP 15 — MAYOR BRECHA DE INVERSIÓN (soles no ejecutados)
        distrito_limpio  año              PIM        Devengado soles_no_ejecutados  pct_ejecucion
                   LIMA 2025 S/ 3,206,134,713 S/ 2,633,767,350      S/ 572,367,363           82.1
                   LIMA 2024 S/ 2,814,948,717 S/ 2,298,876,016      S/ 516,072,701           81.7
                   LIMA 2023 S/ 1,443,207,124 S/ 1,182,961,106      S/ 260,246,018           82.0
             MIRAFLORES 2023   S/ 305,639,797   S/ 190,680,190      S/ 114,959,607           62.4
VILLA MARIA DEL TRIUNFO 2023   S/ 202,286,449   S/ 109,129,014       S/ 93,157,435           53.9
             SAN ISIDRO 2024   S/ 339,603,985   S/ 247,227,288       S/ 92,376,697           72.8
 SAN JUAN DE LURIGANCHO 2023   S/ 366,439,754   S/ 275,895,354       S/ 90,544,400           75.3
             SAN ISIDRO 2023   S/ 288,304,251   S/ 208,452,689       S/ 79,851,562           72.3
             SAN ISIDRO 2025   S/ 362,849,464   S/ 294,774,37

In [15]:
# ============================================
# PROYECTO: Monitor de Eficiencia Municipal
# Autor: José Miguel Ríos Mallasca
# Notebook 02: Análisis SQL con SQLite
# Fecha: Junio 2026
# ============================================

import sqlite3
import pandas as pd

# --- CONEXIÓN ---
ruta_csv = '/content/drive/MyDrive/Proyecto_MuniMetro_Lima /02_Datos_Limpios /lima_metropolitana_final.csv'
df = pd.read_csv(ruta_csv)
conn = sqlite3.connect('/content/drive/MyDrive/Proyecto_MuniMetro_Lima /02_Datos_Limpios /muni_lima.db')
df.to_sql('ejecucion_presupuestal', conn, if_exists='replace', index=False)
print("✅ Base de datos lista — 172 filas cargadas\n")

# --- CONSULTA 1: Top 5 por año ---
print("📊 CONSULTA 1 — TOP 5 POR AÑO")
print("=" * 50)
q1 = """
SELECT año, distrito_limpio, pct_ejecucion,
    RANK() OVER (PARTITION BY año ORDER BY pct_ejecucion DESC) AS ranking
FROM ejecucion_presupuestal
ORDER BY año, ranking LIMIT 20;
"""
print(pd.read_sql_query(q1, conn).to_string(index=False))

# --- CONSULTA 2: Promedio 2023-2025 ---
print("\n📊 CONSULTA 2 — RANKING PROMEDIO 2023-2025")
print("=" * 50)
q2 = """
SELECT distrito_limpio,
    ROUND(AVG(pct_ejecucion), 1) AS promedio_2023_2025,
    MIN(pct_ejecucion) AS minimo,
    MAX(pct_ejecucion) AS maximo
FROM ejecucion_presupuestal
WHERE año IN (2023, 2024, 2025)
GROUP BY distrito_limpio
ORDER BY promedio_2023_2025 DESC;
"""
print(pd.read_sql_query(q2, conn).to_string(index=False))

# --- CONSULTA 3: Eficientes consistentes ---
print("\n📊 CONSULTA 3 — EFICIENTES CONSISTENTES (3 años ≥ 80%)")
print("=" * 50)
q3 = """
SELECT distrito_limpio,
    SUM(CASE WHEN pct_ejecucion >= 80 THEN 1 ELSE 0 END) AS años_sobre_meta,
    ROUND(AVG(pct_ejecucion), 1) AS promedio
FROM ejecucion_presupuestal
WHERE año IN (2023, 2024, 2025)
GROUP BY distrito_limpio
HAVING años_sobre_meta = 3
ORDER BY promedio DESC;
"""
df3 = pd.read_sql_query(q3, conn)
print(f"Total: {len(df3)} distritos\n")
print(df3.to_string(index=False))

# --- CONSULTA 4: Mayor mejora 2023→2025 ---
print("\n📊 CONSULTA 4 — RANKING DE MEJORA 2023 → 2025")
print("=" * 50)
q4 = """
SELECT a.distrito_limpio,
    a.pct_ejecucion AS ejecucion_2023,
    b.pct_ejecucion AS ejecucion_2025,
    ROUND(b.pct_ejecucion - a.pct_ejecucion, 1) AS mejora_puntos
FROM ejecucion_presupuestal a
JOIN ejecucion_presupuestal b ON a.distrito_limpio = b.distrito_limpio
WHERE a.año = 2023 AND b.año = 2025
ORDER BY mejora_puntos DESC;
"""
print(pd.read_sql_query(q4, conn).to_string(index=False))

# --- CONSULTA 5: Brecha de inversión ---
print("\n📊 CONSULTA 5 — MAYOR BRECHA DE INVERSIÓN")
print("=" * 50)
q5 = """
SELECT distrito_limpio, año, PIM, Devengado,
    (PIM - Devengado) AS soles_no_ejecutados,
    pct_ejecucion
FROM ejecucion_presupuestal
WHERE año IN (2023, 2024, 2025)
ORDER BY soles_no_ejecutados DESC LIMIT 15;
"""
df5 = pd.read_sql_query(q5, conn)
df5['PIM'] = df5['PIM'].apply(lambda x: f"S/ {x:,.0f}")
df5['Devengado'] = df5['Devengado'].apply(lambda x: f"S/ {x:,.0f}")
df5['soles_no_ejecutados'] = df5['soles_no_ejecutados'].apply(lambda x: f"S/ {x:,.0f}")
print(df5.to_string(index=False))

print("\n✅ Análisis SQL completo")
conn.close()

✅ Base de datos lista — 172 filas cargadas

📊 CONSULTA 1 — TOP 5 POR AÑO
 año      distrito_limpio  pct_ejecucion  ranking
2023          JESUS MARIA           93.7        1
2023           LURIGANCHO           92.8        2
2023          LA VICTORIA           92.4        3
2023         PUEBLO LIBRE           92.1        4
2023                ANCON           92.1        4
2023           CHORRILLOS           92.0        6
2023                LURIN           92.0        6
2023           SAN MIGUEL           91.7        8
2023    MAGDALENA DEL MAR           90.7        9
2023           LOS OLIVOS           90.4       10
2023            LA MOLINA           89.1       11
2023                COMAS           88.8       12
2023          SANTA ANITA           88.6       13
2023    SANTIAGO DE SURCO           87.7       14
2023 SAN MARTIN DE PORRES           87.7       14
2023        INDEPENDENCIA           87.4       16
2023            SURQUILLO           87.3       17
2023                LINCE  